# PDF 转 Markdown（面向 RAG/Agentic RAG）

把 PDF 转成高质量 Markdown，往往是 RAG（Retrieval-Augmented Generation）系统里**最关键**的一步。

Markdown 的优势：
- **保留结构**：标题层级（# / ## / ###）、列表、引用、代码块等结构信息不丢失，方便检索与回答
- **利于分块**：可以按标题/段落做 Parent/Child chunk，提高召回质量
- **更适合 LLM**：比纯文本更有语义提示，比 JSON 更轻量

> 经验法则：RAG 的上限往往取决于数据抽取（抽得干净/结构保留）做得怎么样。


## 先给你的 PDF 分个类

在写抽取管线前，先回答几个问题：
- PDF 是**可选中文本**，还是**扫描件/图片型 PDF**？
- 有没有**表格**、多栏排版？
- 有没有图表/示意图等视觉信息（是否需要 VLM 才能解释）？

我们把 PDF 分成三个档位：

**🟢 1）简单（数字版文本 PDF）**：正文为主、排版简单、文本可复制

**🟡 2）中等（扫描件/含表格/多栏）**：需要 OCR 与一定的结构化

**🔴 3）复杂（视觉信息重要）**：图表/示意图/版面复杂，需要 VLM（视觉语言模型）逐页解析


## 类别 1：简单 PDF（最快）—— PyMuPDF4LLM

适用：可选中文本的数字版 PDF（不是扫描件）。

工具：
- PyMuPDF4LLM：<https://github.com/pymupdf/PyMuPDF4LLM>

### 安装


In [ ]:
!pip install pymupdf4llm

In [ ]:
import pymupdf4llm
import pathlib

def convert_simple_pdfs_pymupdf4llm(pdf_folder: str, output_folder: str):
    """
    使用 PyMuPDF4LLM 将“文本型 PDF”批量转换为 Markdown。

    参数：
        pdf_folder: PDF 文件夹路径
        output_folder: 输出 Markdown 文件夹路径
    """
    pdf_path = pathlib.Path(pdf_folder)
    output_path = pathlib.Path(output_folder)
    output_path.mkdir(parents=True, exist_ok=True)

    for pdf_file in pdf_path.glob('*.pdf'):
        try:
            md_text = pymupdf4llm.to_markdown(str(pdf_file))
            output_file = output_path / f"{pdf_file.stem}.md"
            output_file.write_text(md_text, encoding='utf-8')
            print(f"✓ Converted: {pdf_file.name}")
        except Exception as e:
            print(f"✗ Error processing {pdf_file.name}: {e}")

    print(f"\n完成！输出在：{output_folder}")

# 示例
# convert_simple_pdfs_pymupdf4llm('./simple_pdfs', './md_output')

## 类别 2：中等复杂度（扫描件/表格/多栏）—— Docling

适用：扫描件、表格较多、排版不规则、需要 OCR 与表格结构化。

工具：Docling（含 OCR 与表格结构输出）
- <https://github.com/DS4SD/docling>

### 安装


In [ ]:
!pip install docling

In [ ]:
from pathlib import Path
from docling.document_converter import DocumentConverter, PdfFormatOption
from docling.datamodel.base_models import InputFormat
from docling.datamodel.pipeline_options import PdfPipelineOptions

def convert_medium_pdfs_docling(pdf_folder: str, output_folder: str):
    """
    使用 Docling 将中等复杂度 PDF 批量转换为 Markdown（含 OCR/表格结构）。
    """
    pipeline_options = PdfPipelineOptions()
    pipeline_options.do_table_structure = True
    pipeline_options.do_ocr = True
    pipeline_options.images_scale = 2.0
    pipeline_options.generate_picture_images = True

    converter = DocumentConverter(
        format_options={
            InputFormat.PDF: PdfFormatOption(pipeline_options=pipeline_options)
        }
    )

    output_path = Path(output_folder)
    output_path.mkdir(parents=True, exist_ok=True)

    for pdf_file in Path(pdf_folder).glob('*.pdf'):
        try:
            result = converter.convert(str(pdf_file))
            markdown_content = result.document.export_to_markdown()
            (output_path / f"{pdf_file.stem}.md").write_text(markdown_content, encoding='utf-8')
            print(f"✓ Converted: {pdf_file.name}")
        except Exception as e:
            print(f"✗ Error processing {pdf_file.name}: {e}")

    print(f"\n完成！输出在：{output_folder}")

# 示例
# convert_medium_pdfs_docling('./medium_pdfs', './md_output')

## 类别 3：复杂 PDF（图表/示意图重要）—— VLM（视觉语言模型）

适用：图表、流程图、示意图等视觉信息对语义非常关键。

思路：逐页渲染为高分辨率图片（如 300 DPI），把图片发给 VLM，让模型输出结构化 Markdown。

优点：结构与视觉理解最好；缺点：速度慢、可能有 API 成本（本地 VLM 也需要较强 GPU）。

下面示例以 Google Gemini 为例（仅示范流程）。


In [ ]:
!pip install PyMuPDF google-genai

In [ ]:
SYSTEM_PROMPT = """你是一个把 PDF 页面转换为 Markdown 的专家。
你的任务：从给定的页面图片中提取所有内容，并输出干净、结构化的 Markdown。
规则：
1) 不要总结，不要补充不存在的内容；只做抽取与格式化。
2) 保留标题层级（# / ## / ###）、列表、代码块等结构。
3) 表格尽量转成 Markdown 表格；不确定就保持可读结构。
4) 对图片/图表：输出简洁但信息充分的描述（不要编造数据）。
只输出 Markdown，不要额外解释。
""".strip()

## 推荐工作流（生产建议）

- 数字版文本 PDF：优先用 PyMuPDF4LLM（快/便宜）
- 扫描件/表格/多栏：优先用 Docling（OCR + 表格结构）或本地 OCR（如 PaddleOCR）
- 图表/复杂视觉：再升级到 VLM

建议先拿 3-5 份代表性 PDF 做小样评测，确认 Markdown 质量（结构、表格、断行）再批处理全量。
